In [48]:
import os
from dotenv import load_dotenv
from utils.utils import inspect, print_indented
from scripts.helpers import (
    read_products,
    EmbeddingClient,
    create_product_text,
    find_n_closest,
)
from scipy.spatial import distance
import numpy as np

load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")
base_url = "https://generativelanguage.googleapis.com/v1beta/openai/"

client = EmbeddingClient(api_key, base_url)

products = read_products()

## Labels


In [49]:
topics = [
    {"label": "Tech"},
    {"label": "Science"},
    {"label": "Sport"},
    {"label": "Business"},
]

# topics = [
#     {"label": "Tech", "description": "Technology, computers, software, hardware, semiconductors, GPUs, chips, artificial intelligence, and innovation"},
#     {"label": "Science", "description": "Scientific research, physics, biology, chemistry, space, and discoveries"},
#     {"label": "Sport", "description": "Sports, athletes, competitions, matches, teams, and championships"},
#     {"label": "Business", "description": "Companies, markets, finance, economy, stocks, revenue, and corporate strategy"},
# ]


In [50]:
class_descriptions = [topic["label"] for topic in topics]
# class_descriptions = [topic["description"] for topic in topics]
class_embeddings = client.create_embeddings(class_descriptions)

## Article to Classify


In [51]:
article = {
    "headline": "How NVIDIA GPUs Could Decide Who Wins the AI Race",
    "keywords": ["ai", "business", "computers"],
}

## Embedding Item to Classify


In [52]:
def create_article_text(article):
    return f"""Headline: {article['headline']}
Keywords: {', '.join(article['keywords'])}
"""

article_text = create_article_text(article)
article_embedding = client.create_embeddings(article_text)[0]

In [53]:
# Compute cosine distances
def find_closest(query_vector, embeddings):
    distances = []
    for index, embedding in enumerate(embeddings):
        dist = distance.cosine(query_vector, embedding)
        distances.append({'distance': dist, 'index': index})
    return min(distances, key=lambda x: x['distance'])

closest = find_closest(article_embedding, class_embeddings)

In [54]:
label = topics[closest['index']]['label']
print(label)

Business


Printing the result, returns the **Business** label. Wait, this doesn't seem right. If we take another look at the article we're classifying, we can see that the headline indicates that the focus of the article is on tech; it's likely that the model captured the business keyword which resulted in the mislabeling. The limitation in our approach that led to this was that the class descriptions lacked detail. The word "Business" or "Tech" doesn't contain much meaning on its own for the model to capture, so a better approach would be to use more detailed class descriptions.

In [55]:
# Add descriptions to labels and run the script again using the description key instead of label
# class_descriptions = [topic["description"] for topic in topics]
# ... rest of code

topics = [
    {"label": "Tech", "description": "Technology, computers, software, hardware, semiconductors, GPUs, chips, artificial intelligence, and innovation"},
    {"label": "Science", "description": "Scientific research, physics, biology, chemistry, space, and discoveries"},
    {"label": "Sport", "description": "Sports, athletes, competitions, matches, teams, and championships"},
    {"label": "Business", "description": "Companies, markets, finance, economy, stocks, revenue, and corporate strategy"},
]


# Example 2: Restaurant Reviews

In [ ]:
sentiments = [{"label": "Positive"}, {"label": "Neutral"}, {"label": "Negative"}]

# sentiments = [
#     {
#         "label": "Positive",
#         "description": "A restaurant review expressing satisfaction and praise: delicious food, excellent service, friendly staff, great atmosphere, would recommend, loved it",
#     },
#     {
#         "label": "Neutral",
#         "description": "A restaurant review that is mixed or indifferent: average, okay, nothing special, decent but unremarkable, neither good nor bad",
#     },
#     {
#         "label": "Negative",
#         "description": "A restaurant review expressing disappointment and complaints: bad food, poor service, rude staff, overpriced, dirty, would not recommend, terrible experience",
#     },
# ]

reviews = [
    "The food was delicious!",
    "The service was a bit slow but the food was good",
    "The food was cold, really disappointing!",
]

In [57]:
# Create a list of class descriptions from the sentiment labels
class_descriptions = [sentiment['label'] for sentiment in sentiments]

# Embed the class_descriptions and reviews
class_embeddings = client.create_embeddings(class_descriptions)
review_embeddings = client.create_embeddings(reviews)

In [58]:
for index, review in enumerate(reviews):
    # Find the closest distance and its index using find_closest()
    closest = find_closest(review_embeddings[index], class_embeddings)
    # Subset sentiments using the index from closest
    label = sentiments[closest["index"]]["label"]
    print(f'"{review}" was classified as {label}')

"The food was delicious!" was classified as Positive
"The service was a bit slow but the food was good" was classified as Neutral
"The food was cold, really disappointing!" was classified as Negative
